# IND320 project 2026
Philip A. K. Øvland \
Github:     https://github.com/Povland/IND320 \
Streamlit:  https://ind320-2026.streamlit.app/\

Video explaining stuff (Please tell me if the link doesn't work):
TO DO


## AI Usage

I used AI as a helping hand with:
    - planning
    - bug fixing
    - making sure I had the correct setup locally
I also used AI for plotting syntax, as the plt-syntax is a headache to remember.

More extensively, AI was use to "auto-complete" things I wrote in VS code. I haven't connected with copilot/etc but still.

## Log - Tasks
TO DO
Firstly we made the entire API-getting etc. into a package. It's the proper python way. We added src\ind320. In the pyproject.toml we now have:

[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"

[tool.hatch.build.targets.wheel]
packages = ["src/ind320"]

### Accounts and repositories
Reuse - done.
Peer review is officially done.

### Local database: Cassandra




Written: 16.10.2026

# Setup
Imports used throughout the notebook. Data access lives in the `ind320` package (`src/ind320/`), installed with `uv sync`.

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

from ind320 import nve

print("Python:", sys.version.split()[0])
print("Running from:", Path.cwd())

NameError: name 'nve' is not defined

# Local database: Cassandra
Cassandra runs locally in Docker:

```
docker run --name my_cassandra -p 9042:9042 -d cassandra:latest   # first time
docker start my_cassandra                                          # afterwards
```

Cassandra needs 30 to 90 seconds to start. The keyspace `ind320_v1` uses `SimpleStrategy` with replication factor 1, since there is only one local node.

In [ ]:
from cassandra.cluster import Cluster

KEYSPACE = "ind320_v1"

cluster = Cluster(["localhost"], port=9042)
session = cluster.connect()

session.execute(f"""
    CREATE KEYSPACE IF NOT EXISTS {KEYSPACE}
    WITH replication = {{'class': 'SimpleStrategy', 'replication_factor': 1}}
""")
session.set_keyspace(KEYSPACE)

session.execute("CREATE TABLE IF NOT EXISTS test (id int PRIMARY KEY, name text)")

print("Cassandra version:", session.execute("SELECT release_version FROM system.local").one().release_version)
print("Keyspaces:", [row.keyspace_name for row in session.execute("SELECT keyspace_name FROM system_schema.keyspaces")])

# Spark
Spark 3.5.1 (PySpark) with the DataStax Spark Cassandra Connector. Requirements on Windows:
- Java 17 (`java -version`)
- Python 3.11 (PySpark 3.5.1 does not support 3.14, which `uv` picked at first)
- `PYSPARK_PYTHON` set to the same Python as the notebook

In [ ]:
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("IND320")
    .config("spark.jars.packages", "com.datastax.spark:spark-cassandra-connector_2.12:3.5.1")
    .config("spark.cassandra.connection.host", "localhost")
    .config("spark.cassandra.connection.port", "9042")
    .getOrCreate()
)
spark

## Test the Spark-Cassandra connection
Write a small DataFrame to Cassandra through Spark, then read it back.

In [ ]:
test_df = spark.createDataFrame([(1, "Oslo"), (2, "Ås")], ["id", "name"])

test_df.write.format("org.apache.spark.sql.cassandra") \
    .options(keyspace=KEYSPACE, table="test") \
    .mode("append") \
    .save()

spark.read.format("org.apache.spark.sql.cassandra") \
    .options(keyspace=KEYSPACE, table="test") \
    .load() \
    .show()

# Remote database: MongoDB
MongoDB. Credentials are stored in `No_sync/atlas-credentials.env`, which is listed in `.gitignore`, with the variables `MONGODB_USERNAME`, `MONGODB_PASSWORD` and `MONGODB_URI`.

In [ ]:
from pymongo import MongoClient


def load_env(path):
    """Reads KEY=value lines from a .env file (handles quotes and a Windows BOM)."""
    values = {}
    for line in Path(path).read_text(encoding="utf-8-sig").splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        sep = "=" if "=" in line else ":"
        key, _, value = line.partition(sep)
        values[key.strip()] = value.strip().strip('"').strip("'")
    return values


creds = load_env("No_sync/atlas-credentials.env")
print(list(creds.keys()))   # names only, never the values

mongo_client = MongoClient(creds["MONGODB_URI"], serverSelectionTimeoutMS=10000)
mongo_client.admin.command("ping")
print("Connected to MongoDB")

mongo_db = mongo_client["ind320"]

## Test that data can be manipulated from Python
Create, read, update and delete on a temporary collection.

In [ ]:
test = mongo_db["test"]

test.insert_many([{"city": "Oslo", "area": "NO1"}, {"city": "Bergen", "area": "NO5"}])
print("After insert:", list(test.find({}, {"_id": 0})))

test.update_one({"city": "Bergen"}, {"$set": {"area": "NO5 (west)"}})
print("After update:", list(test.find({}, {"_id": 0})))

test.delete_one({"city": "Oslo"})
print("After delete:", list(test.find({}, {"_id": 0})))

test.drop()

# API: NVE Magasinstatistikk
- API: https://biapi.nve.no/magasinstatistikk/swagger/index.html
- Description: https://www.nve.no/energi/analyser-og-statistikk/magasinstatistikk/

The CSV from part 1 is replaced by a direct API connection (`ind320.nve`). 
I kept `data/reservoirs.csv` so that `Oblig1.ipynb` still runs.

In [ ]:
# Fetches the data from NVE's API. Renaming headers, datetime conversion,
# index and sorting are done in nve.fetch_reservoir_data()
reservoirs = nve.fetch_reservoir_data()

# Printing data
reservoirs

In [ ]:
# Not all columns are relevant for plotting, so I select the ones I want to plot
columns_to_plot = [
    "filling_degree",
    "filling_TWh",
    "filling_degree_last_week",
    "filling_degree_change"
]

# Normalizing the data to make it easier to compare the different variables
normalized = reservoirs[columns_to_plot].copy()
normalized = (
    normalized - normalized.min()
) / (
    normalized.max() - normalized.min()
)

normalized.plot(
    figsize=(12, 6)
)

plt.title("All variables together")
plt.xlabel("Date")
plt.ylabel("Normalized value")
plt.legend()
plt.show()

In [ ]:
reservoirs[columns_to_plot].plot(
    subplots=True,
    figsize=(12, 20)
)

plt.tight_layout()
plt.show()

In [ ]:
# Capacity by area, because capacity (not by area) is constant over time, we can just take the first value for each area
capacity = reservoirs.groupby("area_nr")["capacity_TWh"].first()
capacity.plot(
    kind="bar",
    figsize=(10, 6),
    title="Reservoir capacity by area",
    xlabel="Area",
    ylabel="Capacity (TWh)"
)

plt.show()